# Line Following — Part 1: Data Collection & Training

This notebook guides you through:
1. Collecting labelled images by **driving the robot with WASD** while the camera saves frames
2. Training a lightweight CNN (MobileNetV2) to predict steering direction
3. Saving the trained model weights as `line_follower.pth`

**Controls during data collection:**
| Key | Action | Label saved |
|-----|--------|-------------|
| W | Forward | `forward` |
| A | Turn left | `left` |
| D | Turn right | `right` |
| S | Stop | *(no image saved)* |

> Images are labelled **by what key you are pressing**, so drive naturally along the track and the labels will be correct automatically.

## Step 1 — Start the Camera

In [2]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import os
from traitlets.config.configurable import SingletonConfigurable

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA  # 672x376
        init_params.camera_fps = 100
        init_params.depth_mode = sl.DEPTH_MODE.NONE         # No depth needed
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open:", repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                bgra = self.image.get_data()
                self.color_value = cv2.cvtColor(bgra, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

camera = Camera()
camera.start()
print(f"Camera started: {camera.width}x{camera.height}")

[2026-04-16 10:20:51 UTC][ZED][INFO] Logging level INFO
[2026-04-16 10:20:51 UTC][ZED][INFO] Logging level INFO
[2026-04-16 10:20:51 UTC][ZED][INFO] Logging level INFO
[2026-04-16 10:20:52 UTC][ZED][INFO] [Init]  Depth mode: NONE
[2026-04-16 10:20:52 UTC][ZED][INFO] [Init]  Camera successfully opened.
[2026-04-16 10:20:52 UTC][ZED][INFO] [Init]  Camera FW version: 1523
[2026-04-16 10:20:52 UTC][ZED][INFO] [Init]  Video mode: VGA@100
[2026-04-16 10:20:52 UTC][ZED][INFO] [Init]  Serial Number: S/N 33683223
Camera started: 672x376


## Step 2 — Collect Training Images with WASD Control

- Click inside the **Input text box** that appears below
- Type WASD keys to drive the robot — **each keypress saves a labelled image**
- Drive the full track multiple times, making sure to cover all sharp turns
- Aim for **at least 50 images per class** (left / forward / right)
- Press **S** to stop the robot (no image saved while stopped)
- When done, run the **Stop Collection** cell below

In [3]:
import ipywidgets as widgets
from IPython.display import display
import motors

# Create dataset folders
CLASS_DIRS = {
    'forward': 'dataset/forward',
    'left':    'dataset/left',
    'right':   'dataset/right',
}
for d in CLASS_DIRS.values():
    os.makedirs(d, exist_ok=True)

robot = motors.MotorsYukon(mecanum=False)

# Counters per class
counts = {'forward': 0, 'left': 0, 'right': 0}

# ── Display widgets ────────────────────────────────────────────────────────────
display_widget = widgets.Image(format='jpeg', width='45%')
count_label    = widgets.Label(value='forward: 0  |  left: 0  |  right: 0')
text_input     = widgets.Text(
    value='',
    placeholder='Click here then type WASD to drive',
    description='Input:',
    disabled=False
)
display(widgets.VBox([display_widget, count_label, text_input]))

# ── Live camera preview ────────────────────────────────────────────────────────
preview_count = 0
def on_camera_change(change):
    global preview_count
    preview_count += 1
    if preview_count % 2 == 0:  # update display every 2nd frame
        frame = change['new']
        if frame is not None:
            preview = cv2.resize(frame, None, fx=0.3, fy=0.3)
            display_widget.value = bgr8_to_jpeg(preview)

camera.observe(on_camera_change, names=['color_value'])

# ── WASD keyboard handler ──────────────────────────────────────────────────────
def save_frame(label):
    """Save the current camera frame to the correct class folder."""
    frame = camera.color_value
    if frame is None:
        return
    idx      = counts[label]
    filename = os.path.join(CLASS_DIRS[label], f'{label}_{idx:05d}.jpg')
    cv2.imwrite(filename, frame)
    counts[label] += 1
    count_label.value = (f"forward: {counts['forward']}  |  "
                         f"left: {counts['left']}  |  "
                         f"right: {counts['right']}")

def on_text_change(change):
    input_value = change['new']
    if not input_value:
        return

    key = input_value[-1].lower()  # use only the most recent character

    if key == 'w':
        robot.forward(0.35)
        save_frame('forward')
    elif key == 'a':
        robot.left(0.30)
        save_frame('left')
    elif key == 'd':
        robot.right(0.30)
        save_frame('right')
    elif key == 's':
        robot.stop()  # stop only — no image saved
    else:
        robot.stop()

text_input.observe(on_text_change, names='value')
print("Ready! Click the Input box and use WASD to drive and collect images.")

Ready! Click the Input box and use WASD to drive and collect images.


### Stop Collection
Run this cell when you have enough images.

In [4]:
camera.unobserve(on_camera_change, names=['color_value'])
text_input.unobserve(on_text_change, names='value')
robot.stop()
print("Collection stopped.")
print(f"  forward : {counts['forward']} images")
print(f"  left    : {counts['left']} images")
print(f"  right   : {counts['right']} images")
print(f"  total   : {sum(counts.values())} images")

Collection stopped.
  forward : 250 images
  left    : 150 images
  right   : 150 images
  total   : 550 images


## Step 3 — Train MobileNetV2

Fine-tunes **MobileNetV2** (pretrained on ImageNet) with a 3-class head.  
Only the classifier layers are trained to keep it fast on the Jetson.

> Expected training time: ~5–10 minutes for 200 images over 10 epochs.

In [3]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split
import torch.nn as nn

DATASET_ROOT = 'dataset'
MODEL_PATH   = 'line_follower.pth'
BATCH_SIZE   = 16
EPOCHS       = 20
LR           = 3e-4
IMG_SIZE     = 224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

# ── Data transforms ───────────────────────────────────────────────────────────
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load dataset
full_dataset = ImageFolder(root=DATASET_ROOT, transform=train_tf)
print(f"Classes (alphabetical order): {full_dataset.classes}")
print(f"Total images: {len(full_dataset)}")

# 80/20 train/val split
n_val   = max(1, int(0.2 * len(full_dataset)))
n_train = len(full_dataset) - n_val
train_set, val_set = random_split(full_dataset, [n_train, n_val])
val_set.dataset = torchvision.datasets.ImageFolder(root=DATASET_ROOT, transform=val_tf)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── Model ─────────────────────────────────────────────────────────────────────
model = torchvision.models.mobilenet_v2(weights='IMAGENET1K_V1')

# Only freeze early layers, let later layers fine-tune
for i, param in enumerate(model.features.parameters()):
    param.requires_grad = True if i > 100 else False

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 3)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted  = outputs.max(1)
        correct       += predicted.eq(labels).sum().item()
        total         += labels.size(0)

    train_acc = 100. * correct / total

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs      = model(images)
            _, predicted = outputs.max(1)
            val_correct  += predicted.eq(labels).sum().item()
            val_total    += labels.size(0)

    val_acc = 100. * val_correct / val_total
    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}]  "
          f"Loss: {running_loss/len(train_loader):.4f}  "
          f"Train Acc: {train_acc:.1f}%  "
          f"Val Acc: {val_acc:.1f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  ✓ Best model saved (val acc: {val_acc:.1f}%)")

print(f"\nTraining complete. Best val accuracy: {best_val_acc:.1f}%")
print(f"Model saved to: {MODEL_PATH}")

Training on: cuda
Classes (alphabetical order): ['forward', 'left', 'right']
Total images: 450


KeyboardInterrupt: 

## Step 4 — Verify the Model

Sanity check — prints predictions vs true labels for a random sample of images.

In [6]:
from PIL import Image as PILImage
import random
from torchvision.datasets import ImageFolder
import os

# Redefine these in case Step 3 wasn't run
DATASET_ROOT = 'dataset'
MODEL_PATH   = 'line_follower.pth'
IMG_SIZE     = 224
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = ImageFolder(root=DATASET_ROOT)
CLASS_NAMES  = full_dataset.classes
CLASS_DIRS   = {
    'forward': 'dataset/forward',
    'left':    'dataset/left',
    'right':   'dataset/right',
}

# Reload model
import torchvision
import torch.nn as nn
model = torchvision.models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 3)
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# Rest of verification
sample_paths = []
for cls_name, cls_dir in CLASS_DIRS.items():
    files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir) if f.endswith('.jpg')]
    if files:
        sample_paths += random.sample(files, min(3, len(files)))

for path in sample_paths:
    img_pil = PILImage.open(path).convert('RGB')
    x = val_tf(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        out   = model(x)
        probs = torch.softmax(out, dim=1)[0]
        pred  = CLASS_NAMES[probs.argmax().item()]
    true_label = os.path.basename(os.path.dirname(path))
    match = '✓' if pred == true_label else '✗'
    print(f"{match} True: {true_label:>8}  |  Pred: {pred:>8}  |  "
          f"Probs: {[f'{p:.2f}' for p in probs.cpu().tolist()]}")

/tmp/ipykernel_13096/813046903.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))


✓ True:  forward  |  Pred:  forward  |  Probs: ['0.95', '0.05', '0.00']
✓ True:  forward  |  Pred:  forward  |  Probs: ['1.00', '0.00', '0.00']
✓ True:  forward  |  Pred:  forward  |  Probs: ['0.99', '0.01', '0.00']
✓ True:     left  |  Pred:     left  |  Probs: ['0.00', '0.99', '0.00']
✓ True:     left  |  Pred:     left  |  Probs: ['0.01', '0.84', '0.15']
✓ True:     left  |  Pred:     left  |  Probs: ['0.00', '1.00', '0.00']
✓ True:    right  |  Pred:    right  |  Probs: ['0.01', '0.03', '0.96']
✓ True:    right  |  Pred:    right  |  Probs: ['0.03', '0.49', '0.49']
✓ True:    right  |  Pred:    right  |  Probs: ['0.06', '0.00', '0.94']


---
## ✅ Done!
Model saved as **`line_follower.pth`**.  
Open **`Line_Following_2_Run.ipynb`** to deploy it on the robot.